# Análisis de una inferencia desde checkpoint

Este notebook ejecuta —o reutiliza desde caché— un rollout greedy de un checkpoint PPO/PPO-GRU. La inferencia queda materializada como traza comprimida, resumen determinista y vídeo opcional.

<div style="border-left: 4px solid #4c78a8; padding: 10px 14px; background: #f4f7fb;">
<b>Alcance:</b> una llamada representa un episodio y una seed. No entrena, no selecciona automáticamente el mejor modelo por métricas y no usa la capa agentic.
</div>

## Requisitos previos

- Entorno instalado con `uv sync --extra examples`.
- Un experimento Navix con al menos un checkpoint guardado.
- Ajustar `EXPERIMENT_NAME` en la sección de configuración.
- La primera ejecución puede tardar por la compilación JAX; las siguientes reutilizan el artefacto si el request es idéntico.

## Qué aprenderás

- Resolver un nodo y un checkpoint exacto del grafo.
- Ejecutar la operation `checkpoint_rollout` con una seed reproducible.
- Inspeccionar el resumen con `summary.as_text()` y `summary.show()`.
- Obtener una vista compacta orientada a LLM con `compact=True`.
- Mostrar el vídeo y el resumen en dos columnas.
- Localizar `rollout.json`, `trace.npz` y `rollout.mp4`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from jarl.experiments.graph import ExperimentGraph
from jarl.experiments.paths import resolve_workspace_root
from jarl.operations.graph.checkpoint_rollout import (
    CheckpointRolloutRequest,
    checkpoint_rollout,
)
from jarl.operations.graph.checkpoints import CheckpointsRequest, checkpoints
from jarl.training.config import RLRunConfig
from jarl.utils import load_project_env

In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "notebook_utils.py").is_file():
    candidate = NOTEBOOK_DIR / "examples" / "notebooks"
    if candidate.is_dir():
        NOTEBOOK_DIR = candidate.resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

load_project_env(start=NOTEBOOK_DIR)

from notebook_utils import display_rollout_analysis

## Configuración del rollout

La identidad de caché incluye checkpoint, seed, entorno, horizonte, config efectivo y perfil de captura. Cambiar cualquiera de estos valores crea un artefacto distinto.

`CHECKPOINT_STEP=None` elige primero el alias `best` y, si no existe, `latest` o `final`. La operation siempre recibe finalmente un step exacto.

In [ ]:
EXPERIMENTS_ROOT = resolve_workspace_root()
EXPERIMENT_NAME = "jarl-example1.3"  # Cambia este nombre por tu experimento.
EXPERIMENT_DIR = EXPERIMENTS_ROOT / EXPERIMENT_NAME

NODE_ID: str | None = None  # None usa el nodo actual.
CHECKPOINT_STEP: int | None = None  # None resuelve best/latest/final.
SEED = 0
ENV_ID: str | None = None  # None conserva el entorno del nodo.
MAX_STEPS: int | None = None  # None usa el horizonte del entorno.
RECORD_VIDEO = True
VIDEO_VIEW_MODE = "full"  # "full" o "first_person".

print(f"Experimentos: {EXPERIMENTS_ROOT}")
print(f"Experimento:  {EXPERIMENT_DIR}")

## Seleccionar nodo y checkpoint

Primero inspeccionamos el registro de checkpoints. El alias solo ayuda a elegir; la inferencia se materializa contra el step numérico resultante.

In [ ]:
if not (EXPERIMENT_DIR / "experiment.json").is_file():
    raise FileNotFoundError(
        f"No existe un experimento JARL en {EXPERIMENT_DIR}. Ajusta EXPERIMENT_NAME o EXPERIMENT_DIR."
    )

graph = ExperimentGraph.from_directory(EXPERIMENT_DIR, config_cls=RLRunConfig)
workspace = graph.get_node(NODE_ID) if NODE_ID else graph.current_node
config = graph.resolve_config(workspace)
overview = checkpoints(graph, CheckpointsRequest(node_id=workspace.id))

selected = overview.best or overview.latest or overview.final
checkpoint_step = CHECKPOINT_STEP
if checkpoint_step is None:
    if selected is None:
        raise ValueError(f"El nodo {workspace.id!r} no tiene checkpoints seleccionables.")
    checkpoint_step = selected.checkpoint_step

print(f"Nodo:       {workspace.id}")
print(f"Algoritmo:  {config.algorithm.name}")
print(f"Entorno:    {config.environment.env_id}")
print(f"Checkpoint: {checkpoint_step}")

## Ejecutar o reutilizar la inferencia

La operation valida el checkpoint, la compatibilidad del entorno y el horizonte antes de compilar. Con `RECORD_VIDEO=True`, las matrices analíticas y los frames RGB se capturan en el mismo rollout; no se ejecuta un segundo episodio para generar el MP4.

In [ ]:
response = checkpoint_rollout(
    graph,
    CheckpointRolloutRequest(
        node_id=workspace.id,
        checkpoint_step=checkpoint_step,
        seed=SEED,
        env_id=ENV_ID,
        max_steps=MAX_STEPS,
        record_video=RECORD_VIDEO,
        video_view_mode=VIDEO_VIEW_MODE,
    ),
)

print(f"Rollout:   {response.rollout_id}")
print(f"Cache hit: {response.cache_hit}")
if response.rollout_seconds is not None:
    print(f"Rollout:   {response.rollout_seconds:.3f} s")
if response.transfer_seconds is not None:
    print(f"Transfer:  {response.transfer_seconds:.3f} s")

## Vídeo y resumen lado a lado

El panel izquierdo reproduce el MP4 persistido. El derecho resume outcome, acciones, eventos e intervalos de visibilidad derivados de la misma traza.

In [ ]:
video_relative = next(
    (path for path in response.summary.artifact_paths if path.endswith("rollout.mp4")),
    None,
)
video_path = workspace.path / video_relative if video_relative else None

display_rollout_analysis(
    response.summary,
    video_path,
    cache_hit=response.cache_hit,
)

## Resumen legible y vista LLM

`NavixRolloutSummary` expone `as_text()` para texto plano y `show()` para mostrarlo en Jupyter. Por defecto el formato es humano; con `compact=True` devuelve una vista estable para tools o prompts.

In [ ]:
response.summary.show()


In [ ]:

print("\n--- Vista compacta para LLM ---\n")
print(response.summary.as_text(compact=True))

## Conclusiones

- La seed se usa directamente, por lo que un mismo checkpoint y request son reproducibles.
- Vídeo y telemetría analítica proceden de un único rollout.
- `rollout_id` identifica exactamente checkpoint, target, horizonte, config y perfil de captura.
- `summary.as_text()` evita conocer todos los campos del dataclass; `compact=True` prepara el payload para una LLM.
- La evidencia completa sigue en `trace.npz` para análisis posterior.
- Repetir la misma celda reutiliza el artefacto validado.

## Referencias

- [`jarl.operations.graph.checkpoint_rollout`](../../src/jarl/operations/graph/checkpoint_rollout.py)
- [`jarl.inference.run`](../../src/jarl/inference/run.py)
- [Persistencia de rollouts](../../src/jarl/experiments/io/rollouts.py)
- [Telemetría Navix](../../src/jarl/envs/navix/telemetry/)
- [Inventario de datos PPO](../../docs/development/ppo-collected-data.md)